# Parallel Workflow: Multi-Perspective Paper Review

This notebook demonstrates a parallel workflow with fan-out and fan-in.

Scenario:

A paper is reviewed from different perspectives:

- Methodology
- Related work
- Clarity and structure
- Ethics and limitations

The reviews can be performed independently and then combined.

## Setup

In [8]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)



from picoagents.workflow import Workflow, WorkflowRunner, FunctionStep
from picoagents.workflow.core import WorkflowMetadata, StepMetadata, Context

runner = WorkflowRunner()

API key loaded successfully.


## Define models and step functions

In [9]:
class PaperInput(BaseModel):
    """Input schema containing the paper metadata to review."""
    title: str
    abstract: str


class ReviewOutput(BaseModel):
    """Output from one review perspective branch."""
    perspective: str
    comments: List[str]


class CombinedReviewOutput(BaseModel):
    """Final combined review text returned by the fan-in step."""
    result: str


async def prepare_paper(input_data: PaperInput, context: Context) -> PaperInput:
    """Store shared paper metadata in context before parallel branches."""
    context.set("paper_title", input_data.title)
    return input_data


async def review_methodology(input_data: PaperInput, context: Context) -> ReviewOutput:
    """Generate methodology-focused review comments."""
    return ReviewOutput(
        perspective="Methodology",
        comments=[
            "Clarify the experimental design.",
            "Explain dataset selection and evaluation metrics."
        ]
    )


async def review_related_work(input_data: PaperInput, context: Context) -> ReviewOutput:
    """Generate related-work-focused review comments."""
    return ReviewOutput(
        perspective="Related Work",
        comments=[
            "Add more recent references.",
            "Clarify the research gap."
        ]
    )


async def review_clarity(input_data: PaperInput, context: Context) -> ReviewOutput:
    """Generate clarity-and-structure review comments."""
    return ReviewOutput(
        perspective="Clarity and Structure",
        comments=[
            "Improve transitions between sections.",
            "Define key terms earlier."
        ]
    )


async def review_ethics(input_data: PaperInput, context: Context) -> ReviewOutput:
    """Generate ethics-and-limitations review comments."""
    return ReviewOutput(
        perspective="Ethics and Limitations",
        comments=[
            "Discuss data privacy risks.",
            "Add limitations about generalizability."
        ]
    )


async def combine_reviews(input_data: ReviewOutput, context: Context) -> CombinedReviewOutput:
    """Produce the final combined-review summary in the fan-in step."""
    # Depending on your workflow runner implementation, fan-in may pass one branch output
    # or provide access to all step outputs through execution metadata.
    # This simple version demonstrates the target final step.
    title = context.get("paper_title", "Unknown paper")
    return CombinedReviewOutput(
        result=(
            f"Combined review for: {title}\n"
            f"Received final branch perspective: {input_data.perspective}\n"
            "In a production implementation, collect all branch outputs before composing the final review."
        )
    )

## Build parallel workflow

### Workflow Diagram: How add_step and add_edge map to the graph

`add_step(...)` creates nodes.
`add_edge(from, to)` creates arrows (dependencies) between nodes.

```mermaid
flowchart
    A[prepare_paper] --> B[review_methodology]
    A --> C[review_related_work]
    A --> D[review_clarity]
    A --> E[review_ethics]
    B --> F[combine_reviews]
    C --> F
    D --> F
    E --> F
```

Quick mapping to code:
- `add_step(...)`: registers each node (`prepare`, 4 reviewers, `combine`).
- `add_edge(...)`: wires execution order (`prepare -> reviewers -> combine`).
- `set_start_step("prepare_paper")`: sets entry point for execution.

In [10]:
prepare_step = FunctionStep(
    step_id="prepare_paper",
    metadata=StepMetadata(name="Prepare Paper"),
    input_type=PaperInput,
    output_type=PaperInput,
    func=prepare_paper
)

methodology_step = FunctionStep(
    step_id="review_methodology",
    metadata=StepMetadata(name="Review Methodology"),
    input_type=PaperInput,
    output_type=ReviewOutput,
    func=review_methodology
)

related_work_step = FunctionStep(
    step_id="review_related_work",
    metadata=StepMetadata(name="Review Related Work"),
    input_type=PaperInput,
    output_type=ReviewOutput,
    func=review_related_work
)

clarity_step = FunctionStep(
    step_id="review_clarity",
    metadata=StepMetadata(name="Review Clarity"),
    input_type=PaperInput,
    output_type=ReviewOutput,
    func=review_clarity
)

ethics_step = FunctionStep(
    step_id="review_ethics",
    metadata=StepMetadata(name="Review Ethics"),
    input_type=PaperInput,
    output_type=ReviewOutput,
    func=review_ethics
)

combine_step = FunctionStep(
    step_id="combine_reviews",
    metadata=StepMetadata(name="Combine Reviews"),
    input_type=ReviewOutput,
    output_type=CombinedReviewOutput,
    func=combine_reviews
)

parallel_workflow = (
    Workflow(metadata=WorkflowMetadata(name="Parallel Multi-Perspective Paper Review"))
    # add_step: register each node in the workflow graph
    .add_step(prepare_step)
    .add_step(methodology_step)
    .add_step(related_work_step)
    .add_step(clarity_step)
    .add_step(ethics_step)
    .add_step(combine_step)
    # add_edge: define dependencies and execution flow between nodes
    .add_edge("prepare_paper", "review_methodology")
    .add_edge("prepare_paper", "review_related_work")
    .add_edge("prepare_paper", "review_clarity")
    .add_edge("prepare_paper", "review_ethics")
    .add_edge("review_methodology", "combine_reviews")
    .add_edge("review_related_work", "combine_reviews")
    .add_edge("review_clarity", "combine_reviews")
    .add_edge("review_ethics", "combine_reviews")
    .set_start_step("prepare_paper")
)

## Run example

In [11]:
paper = {
    "title": "AI Tutors for Supporting Machine Learning Education",
    "abstract": "This paper proposes an AI tutor for supporting students in machine learning courses."
}

async for event in runner.run_stream(parallel_workflow, paper):
    print(event)

[21:01:39] 🚀 Workflow started with input: {'title': 'AI Tutors for Supporting Machine Learning Education', 'abstract': 'This paper proposes an AI tutor for supporting students in machine learning courses.'}
[21:01:39] ▶️  Step 'prepare_paper' started
[21:01:39] ✅ Step 'prepare_paper' completed → {'title': 'AI Tutors for Supporting Machine Learning Education', 'abstract': 'This paper proposes an AI tutor for supporting students in machine learning courses.'}
[21:01:39] 🔗 prepare_paper → review_methodology
[21:01:39] 🔗 prepare_paper → review_related_work
[21:01:39] 🔗 prepare_paper → review_clarity
[21:01:39] 🔗 prepare_paper → review_ethics
[21:01:39] ▶️  Step 'review_methodology' started
[21:01:39] ▶️  Step 'review_related_work' started
[21:01:39] ▶️  Step 'review_clarity' started
[21:01:39] ▶️  Step 'review_ethics' started
[21:01:39] ✅ Step 'review_ethics' completed → {'perspective': 'Ethics and Limitations', 'comments': ['Discuss data privacy risks.', 'Add limitations about generalizab

## Student Exercises

Task 1: Add a new parallel branch review_reproducibility and include it in fan-in.

```python
# Your code is here
# End of code
```

Task 2: Implement deterministic fan-in ordering by perspective name before composing final output.

```python
# Your code is here
# End of code
```

Task 3: Include a summary score (0-100) in CombinedReviewOutput based on number of actionable comments.

```python
# Your code is here
# End of code
```

## Solutions

Solution 1

```python
async def review_reproducibility(input_data: PaperInput, context: Context) -> ReviewOutput:
    return ReviewOutput(
        perspective="Reproducibility",
        comments=[
            "Specify random seeds and environment versions.",
            "Share preprocessing and split protocol in detail.",
        ],
    )

repro_step = FunctionStep(
    step_id="review_reproducibility",
    metadata=StepMetadata(name="Review Reproducibility"),
    input_type=PaperInput,
    output_type=ReviewOutput,
    func=review_reproducibility,
 )

# Add edges: prepare_paper -> review_reproducibility -> combine_reviews
```

Solution 2

```python
all_reviews = context.get("all_reviews", [])
all_reviews.append(input_data.model_dump())
context.set("all_reviews", all_reviews)

ordered = sorted(all_reviews, key=lambda x: x["perspective"])
```

Solution 3

```python
class CombinedReviewOutput(BaseModel):
    result: str
    score: int

actionable_count = sum(len(r["comments"]) for r in ordered)
score = min(100, actionable_count * 10)
return CombinedReviewOutput(result=summary_text, score=score)
```